# NoSQL Landing to Bronze

Papermill-compatible notebook that reads `reviews_nosql.json` from MinIO Landing and writes the NoSQL source as a Delta table in the Bronze bucket. The object and table names remain `reviews_nosql` for pipeline compatibility, while the current document shape is support tickets.

In [ ]:
from pathlib import Path

project_root = str(Path.cwd())
input_bucket = None
input_prefix = "nosql"
input_key = None
output_bucket = None
output_prefix = "reviews_nosql"
write_mode = "overwrite"
minio_endpoint = None
minio_access_key = None
minio_secret_key = None
app_name = "01b_nosql_to_bronze"
spark_master = "local[*]"


In [ ]:
import ctypes
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from urllib.parse import urlparse

import boto3
from botocore.exceptions import BotoCoreError, ClientError


In [ ]:
DEFAULT_INPUT_BUCKET = "landing"
DEFAULT_INPUT_PREFIX = "nosql"
DEFAULT_INPUT_OBJECT_NAME = "reviews_nosql.json"
DEFAULT_OUTPUT_BUCKET = "bronze"
DEFAULT_OUTPUT_PREFIX = "reviews_nosql"
DEFAULT_WRITE_MODE = "overwrite"
DEFAULT_SPARK_MASTER = "local[*]"
HADOOP_AWS_PACKAGE = "org.apache.hadoop:hadoop-aws:3.3.4"
TIMESTAMP_PATTERN = "yyyy-MM-dd'T'HH:mm:ssX"


@dataclass(frozen=True)
class StorageConfig:
    endpoint_url: str
    access_key: str
    secret_key: str
    input_bucket: str
    input_prefix: str
    output_bucket: str
    output_prefix: str


In [ ]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        return

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def value_or_env(value: str | None, env_key: str, default: str) -> str:
    if value not in (None, ""):
        return str(value)
    return os.getenv(env_key, default)


def normalize_prefix(prefix: str) -> str:
    return prefix.strip("/")


def build_storage_config() -> StorageConfig:
    return StorageConfig(
        endpoint_url=value_or_env(minio_endpoint, "MINIO_ENDPOINT", "http://localhost:9000"),
        access_key=value_or_env(minio_access_key, "MINIO_ACCESS_KEY", "minioadmin"),
        secret_key=value_or_env(minio_secret_key, "MINIO_SECRET_KEY", "minioadmin"),
        input_bucket=value_or_env(input_bucket, "MINIO_BUCKET_LANDING", DEFAULT_INPUT_BUCKET),
        input_prefix=normalize_prefix(input_prefix or DEFAULT_INPUT_PREFIX),
        output_bucket=value_or_env(output_bucket, "MINIO_BUCKET_BRONZE", DEFAULT_OUTPUT_BUCKET),
        output_prefix=normalize_prefix(output_prefix or DEFAULT_OUTPUT_PREFIX),
    )


def build_s3_client(config: StorageConfig):
    return boto3.client(
        "s3",
        endpoint_url=config.endpoint_url,
        aws_access_key_id=config.access_key,
        aws_secret_access_key=config.secret_key,
        region_name="us-east-1",
    )


def ensure_bucket_exists(s3_client, bucket_name: str) -> None:
    try:
        s3_client.head_bucket(Bucket=bucket_name)
    except ClientError as exc:
        error_code = str(exc.response.get("Error", {}).get("Code", ""))
        if error_code in {"404", "NoSuchBucket", "NotFound"}:
            s3_client.create_bucket(Bucket=bucket_name)
            return
        raise


In [ ]:
def resolve_input_key(s3_client, bucket_name: str, prefix: str, requested_key: str | None) -> str:
    if requested_key:
        return requested_key.lstrip("/")

    normalized_prefix = f"{prefix}/" if prefix else ""
    paginator = s3_client.get_paginator("list_objects_v2")
    matches: list[dict] = []

    for page in paginator.paginate(Bucket=bucket_name, Prefix=normalized_prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(DEFAULT_INPUT_OBJECT_NAME):
                matches.append(obj)

    if not matches:
        raise FileNotFoundError(
            f"Could not find '{DEFAULT_INPUT_OBJECT_NAME}' in bucket '{bucket_name}' "
            f"under prefix '{normalized_prefix or '/'}'."
        )

    matches.sort(key=lambda item: item["LastModified"], reverse=True)
    return matches[0]["Key"]


def build_s3a_uri(bucket_name: str, key_or_prefix: str) -> str:
    normalized = key_or_prefix.strip("/")
    if not normalized:
        return f"s3a://{bucket_name}"
    return f"s3a://{bucket_name}/{normalized}"


def parse_minio_endpoint(endpoint_url: str) -> tuple[str, bool]:
    parsed = urlparse(endpoint_url if "://" in endpoint_url else f"http://{endpoint_url}")
    endpoint = parsed.netloc or parsed.path
    ssl_enabled = parsed.scheme == "https"
    return endpoint, ssl_enabled


In [ ]:
def append_java_tool_option(option: str) -> None:
    current_value = os.environ.get("JAVA_TOOL_OPTIONS", "")
    if option not in current_value.split():
        os.environ["JAVA_TOOL_OPTIONS"] = " ".join(part for part in [current_value, option] if part).strip()


def resolve_windows_short_path(path: Path) -> Path:
    buffer_size = 1024
    output_buffer = ctypes.create_unicode_buffer(buffer_size)
    result = ctypes.windll.kernel32.GetShortPathNameW(str(path), output_buffer, buffer_size)
    if result == 0:
        return path
    return Path(output_buffer.value)


def prepare_local_spark_environment() -> None:
    try:
        import pyspark
    except ModuleNotFoundError as exc:
        raise RuntimeError("pyspark is required to prepare the local Spark environment.") from exc

    spark_home = Path(pyspark.__file__).resolve().parent
    python_executable = Path(sys.executable).resolve()

    if os.name == "nt":
        append_java_tool_option("-Djavax.net.ssl.trustStoreType=Windows-ROOT")
        default_hadoop_home = Path("C:/hadoop")
        if "HADOOP_HOME" not in os.environ and (default_hadoop_home / "bin" / "winutils.exe").exists():
            os.environ["HADOOP_HOME"] = str(default_hadoop_home)
            os.environ["hadoop.home.dir"] = str(default_hadoop_home)
        spark_home = resolve_windows_short_path(spark_home)
        python_executable = resolve_windows_short_path(python_executable)

    os.environ["SPARK_HOME"] = str(spark_home)
    os.environ.setdefault("PYSPARK_PYTHON", str(python_executable))
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", str(python_executable))


def build_spark_session(config: StorageConfig):
    try:
        from delta import configure_spark_with_delta_pip
        from pyspark.sql import SparkSession
    except ModuleNotFoundError as exc:
        raise RuntimeError("pyspark and delta-spark are required to run this notebook.") from exc

    prepare_local_spark_environment()
    endpoint, ssl_enabled = parse_minio_endpoint(config.endpoint_url)
    builder = configure_spark_with_delta_pip(
        SparkSession.builder.appName(app_name)
        .master(spark_master or DEFAULT_SPARK_MASTER)
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.endpoint", endpoint)
        .config("spark.hadoop.fs.s3a.access.key", config.access_key)
        .config("spark.hadoop.fs.s3a.secret.key", config.secret_key)
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(ssl_enabled).lower())
        .config("spark.hadoop.fs.s3a.fast.upload", "true")
        .config("spark.hadoop.fs.s3a.fast.upload.buffer", "array"),
        extra_packages=[HADOOP_AWS_PACKAGE],
    )
    return builder.getOrCreate()


In [ ]:
def parse_timestamp_if_present(df, column_name: str):
    from pyspark.sql import functions as F

    if column_name not in df.columns:
        return df
    return df.withColumn(column_name, F.to_timestamp(column_name, TIMESTAMP_PATTERN))


def read_nosql_dataframe(spark, input_path: str):
    from pyspark.sql import functions as F

    source_df = spark.read.option("multiLine", True).json(input_path)
    source_df = parse_timestamp_if_present(source_df, "created_at")
    source_df = parse_timestamp_if_present(source_df, "opened_at")
    source_df = parse_timestamp_if_present(source_df, "closed_at")
    return (
        source_df.withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file", F.input_file_name())
    )


In [ ]:
def detect_document_shape(df) -> str:
    columns = set(df.columns)
    if {"ticket_id", "issue_type", "messages", "agent", "resolution"}.issubset(columns):
        return "support_tickets"
    if {"review_id", "sentiment", "comment_text", "tags", "created_at"}.issubset(columns):
        return "reviews"
    raise ValueError(f"Unsupported NoSQL document shape. Columns found: {sorted(columns)}")


def validate_bronze_dataframe(df) -> str:
    from pyspark.sql import functions as F
    from pyspark.sql.types import ArrayType, StructType, TimestampType

    if not df.take(1):
        raise ValueError("The NoSQL source produced zero rows.")

    field_map = {field.name: field.dataType for field in df.schema.fields}
    metadata_columns = {"_ingestion_timestamp", "_source_file"}
    missing_metadata = metadata_columns.difference(field_map)
    if missing_metadata:
        raise ValueError(f"Missing expected Bronze metadata columns: {sorted(missing_metadata)}")
    if not isinstance(field_map["_ingestion_timestamp"], TimestampType):
        raise TypeError("Column '_ingestion_timestamp' must be stored as TimestampType.")

    shape = detect_document_shape(df)
    if shape == "support_tickets":
        expected_columns = {
            "ticket_id",
            "order_id",
            "customer_id",
            "channel",
            "issue_type",
            "priority",
            "status",
            "opened_at",
            "agent",
            "resolution",
            "messages",
        }
        missing_columns = expected_columns.difference(field_map)
        if missing_columns:
            raise ValueError(f"Missing expected support-ticket columns: {sorted(missing_columns)}")
        if not isinstance(field_map["opened_at"], TimestampType):
            raise TypeError("Column 'opened_at' must be stored as TimestampType in Bronze.")
        if "closed_at" in field_map and not isinstance(field_map["closed_at"], TimestampType):
            raise TypeError("Column 'closed_at' must be stored as TimestampType in Bronze.")
        if not isinstance(field_map["agent"], StructType):
            raise TypeError("Column 'agent' must be preserved as StructType in Bronze.")
        if not isinstance(field_map["resolution"], StructType):
            raise TypeError("Column 'resolution' must be preserved as StructType in Bronze.")
        if not isinstance(field_map["messages"], ArrayType):
            raise TypeError("Column 'messages' must be preserved as ArrayType in Bronze.")
        invalid_rows = (
            df.filter(
                F.col("ticket_id").isNull()
                | F.col("order_id").isNull()
                | F.col("customer_id").isNull()
                | F.col("opened_at").isNull()
                | F.col("_source_file").isNull()
                | F.col("messages").isNull()
                | (F.size("messages") == 0)
            )
            .limit(1)
            .count()
        )
        if invalid_rows:
            raise ValueError("The Bronze dataset contains invalid required support-ticket fields.")
        return shape

    if not isinstance(field_map["created_at"], TimestampType):
        raise TypeError("Column 'created_at' must be stored as TimestampType in Bronze.")
    if not isinstance(field_map["tags"], ArrayType):
        raise TypeError("Column 'tags' must be preserved as ArrayType in Bronze.")
    invalid_rows = (
        df.filter(
            F.col("review_id").isNull()
            | F.col("order_id").isNull()
            | F.col("customer_id").isNull()
            | F.col("created_at").isNull()
            | F.col("_source_file").isNull()
        )
        .limit(1)
        .count()
    )
    if invalid_rows:
        raise ValueError("The Bronze dataset contains invalid required review fields.")
    return shape


In [ ]:
def write_bronze_delta(df, output_path: str, mode: str) -> None:
    if mode not in {"overwrite", "append"}:
        raise ValueError("write_mode must be 'overwrite' or 'append'.")
    df.write.format("delta").mode(mode).option("overwriteSchema", "true").save(output_path)


def read_back_delta(spark, output_path: str):
    return spark.read.format("delta").load(output_path)


def print_summary(input_path: str, output_path: str, row_count: int, document_shape: str) -> None:
    summary = {
        "input_path": input_path,
        "output_path": output_path,
        "rows_written": row_count,
        "bronze_table": output_prefix or DEFAULT_OUTPUT_PREFIX,
        "document_shape": document_shape,
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
load_env_file(Path(project_root) / ".env")
config = build_storage_config()
resolved_write_mode = write_mode or DEFAULT_WRITE_MODE
requested_input_key = input_key if input_key not in (None, "") else None

s3_client = build_s3_client(config)
ensure_bucket_exists(s3_client, config.output_bucket)

try:
    resolved_input_key = resolve_input_key(
        s3_client=s3_client,
        bucket_name=config.input_bucket,
        prefix=config.input_prefix,
        requested_key=requested_input_key,
    )
except (BotoCoreError, ClientError) as exc:
    raise RuntimeError(f"Failed to resolve the NoSQL JSON in Landing: {exc}") from exc

input_path = build_s3a_uri(config.input_bucket, resolved_input_key)
output_path = build_s3a_uri(config.output_bucket, config.output_prefix)

spark = build_spark_session(config)
bronze_df = read_nosql_dataframe(spark, input_path)
document_shape = validate_bronze_dataframe(bronze_df)
row_count = bronze_df.count()

write_bronze_delta(bronze_df, output_path, resolved_write_mode)

persisted_df = read_back_delta(spark, output_path)
persisted_document_shape = validate_bronze_dataframe(persisted_df)
if persisted_document_shape != document_shape:
    raise ValueError("Persisted Delta table shape differs from the source DataFrame shape.")
print_summary(input_path, output_path, row_count, document_shape)
